In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
import os
import sys
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 500)
import importlib
import os
import sys
#root_path = os.path.dirname(os.path.dirname(os.path.abspath(os.path.dirname('__file__'))))

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)
from env.parameters import P
from analysis_functions.data_preparation import cohort_type_adjustment
import dask.dataframe as dd
import pickle
import yaml
from analysis_functions.feature_engineering import (
keep_england_country_imd,
drop_unknown_country_imd,
imd_quantiles,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
find_normal_boundaries, 
find_skewed_boundaries,
diagnostic_plots,
plot_boxplot_and_hist,
outlier_analysis,
keep_england_country_imd,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
impute_nulls_mice,
create_age_bands
)

from analysis_functions.custom_transformers import (
Custom_Winsoriser,
bmi_categoriser,
fev1fvc_ratio_categoriser,
traffic_intensity_quantiles,
inverse_distance_quantiles,
CustomFrequencyBinner,
CustomWaistBinner,
MultiTransform,
MultiTransformList,
CustomBMICategoriser,
CustomFev1FvcRatioCategoriser,
CustomInverseDistanceCategoriser,
CustomTrafficIntensityCategoriser,
ColumnSelector,
CustomBinaryCategoriserAroundMean,
CustomBinaryCategoriserAroundMedian,
CustomBinaryCategoriserAroundDecile
)
from scipy.stats.mstats import winsorize
from fancyimpute import IterativeImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer
from scipy.stats import shapiro
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import warnings
from diffprivlib.utils import PrivacyLeakWarning
from sklearn.decomposition import PCA
import diffprivlib as dp
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
accuracy_score, confusion_matrix, 
classification_report, f1_score, 
roc_curve, roc_auc_score,
precision_recall_curve,
precision_score, recall_score, average_precision_score,balanced_accuracy_score, matthews_corrcoef)
import shap
from scipy.stats import chi2
from pipeline_functions import *
from custom_plots import *

from sklearn.neighbors import NearestNeighbors
from scipy.stats import fisher_exact, norm, chi2_contingency
from statsmodels.stats.contingency_tables import Table2x2
from epi_functions import *

In [ ]:
cohort_path = P.cohorts_ukb_start_gphesonly_path

In [ ]:
# Load the cohort
df_in = pd.read_csv(f'''{P.cohorts_ukb_start_gphesonly_path}/analysis_csv/epi_analysis_ready_df_ukb_start_gphesonly.csv''')

In [ ]:
# Load column types
pickle_file = f'''{P.cohorts_ukb_start_gphesonly_path}/pickle/cols_dict_gphesonly_all.pickle'''

with open(pickle_file, 'rb') as f:
     cols_dict = pickle.load(f)

print(cols_dict.keys())

In [ ]:
print(df_in.shape)



In [ ]:
# Assign types to the dataframe
df_in = cohort_type_adjustment(df_in, cols_dict)




In [ ]:
df_in["country_imd"].value_counts(dropna=False)

In [ ]:
df_in.shape

# How many with less than 1 year of follow up?

In [ ]:
# Outcome of interest
col_o = "flag_post_cohort_start_exac_ocs_y1"

In [ ]:
df_in[df_in["follow_up_asthma_pre_cohort_start"]<1].shape[0]

In [ ]:
df_in[df_in["follow_up_asthma_pre_cohort_start"]<1]["follow_up_asthma_pre_cohort_start"].describe()

In [ ]:
# Drop these
df = df_in[df_in["follow_up_asthma_pre_cohort_start"]>=1]

In [ ]:
# Any outcome on study start?

df[df['evdt_first_post_cohort_start_exac']==df['evdt_cohort_start']]

In [ ]:
print(df_in.shape)
print(df.shape)

In [ ]:
df_in.shape[0] - df.shape[0]

In [ ]:
df[col_o].value_counts()/df.shape[0]*100

# Asthma onset (40)

https://www.jaci-inpractice.org/article/S2213-2198(22)00330-0/fulltext


In [ ]:

df.loc[:, "late_onset_asthma_40"] = df["age_asthma"].apply(lambda x: 1 if x >=40 else 0)

In [ ]:
df["late_onset_asthma_40"].value_counts()

# cardinal symptom

In [ ]:
df.loc[:, "cardinal_symptoms"] =df[["wheeze_field", "shortness_breath_field", "chest_pain_field"]].any(axis=1).astype(int)

In [ ]:
df["cardinal_symptoms"].value_counts()

# How many deaths before the outcome (do not censor). Just do sensitivity without these later

In [ ]:
df[["evdt_cohort_start","evdt_first_post_cohort_start_exac", "dod"]].dtypes

In [ ]:
#condition = (df[col_o]==1) & (df['dod'].notnull()) & ((df['dod'] >= df['evdt_first_post_cohort_start_exac']) & (df['dod'] <= df['evdt_first_post_cohort_start_exac'] + pd.Timedelta(days=365)))
condition = (df['dod'].notnull()) & (df['dod'] > df['evdt_cohort_start']) & (df['dod'] <= df['evdt_cohort_start'] + pd.Timedelta(days=365))

df[condition].shape

In [ ]:
# any exac in these?
df[condition & df[col_o]==1][["evdt_cohort_start", "evdt_first_post_cohort_start_exac", "dod"]].shape

In [ ]:
df['tte_cohort_start_to_exac'].describe()

In [ ]:
# any deaths recorded before exac 1 event data? any deaths in non-exac within a year?

df[df['dod']<df['evdt_first_post_cohort_start_exac']]

# OHE Smoking and Ethnicity

- We have not dropped any level here. Drop in modeling

In [ ]:
df = pd.get_dummies(df, columns=['eth_grouped_1b', 'desc_smoking_at_baseline'], drop_first=False)
dummy_columns = [col for col in df.columns if 'eth_grouped_1b_' in col or 'desc_smoking_at_baseline_' in col]
df[dummy_columns] = df[dummy_columns].astype(int)

In [ ]:
df.columns

In [ ]:
# Random seeds for non_private and differentially private models
np_random_seed = 7
dp_random_seed = 47

In [ ]:
# Covariates
cov_list= ['age_60+', 'sex_female', 
           'eth_non_white',
           'late_onsetasthma_40',
            'pheno_anxiety_pre_cohort_start',
           'bmi_30_imputed', 
           'pheno_ckd_pre_cohort_start',
           'pheno_copd_pre_cohort_start',
           'pheno_cvd_pre_cohort_start',
            'pheno_depression_pre_cohort_start',
           'pheno_diabetes_pre_cohort_start',
           'pheno_ht_pre_cohort_start',
           'cardinal_symptoms',
            'flag_pre_cohort_start_exac_y1', 'flag_pre_cohort_start_meds_ocs_y1',
                  'desc_smoking_at_baseline_Current',
       'desc_smoking_at_baseline_Previous'] 



In [ ]:
# Rename covariates
rename_dict = {
 'age_60+': "Age≥60",
 'sex_female': "Sex_female",
 'eth_non_white': "Non_white",
 'late_onset_asthma_40': "Late_onset",
 'pheno_anxiety_pre_cohort_start': "Anxiety",
 'bmi_30_imputed': "BMI≥30",
 'pheno_ckd_pre_cohort_start': "CKD",
 'pheno_copd_pre_cohort_start' :"COPD",
 'pheno_cvd_pre_cohort_start': "CVD",
 'pheno_depression_pre_cohort_start': "Depression",
 'pheno_diabetes_pre_cohort_start': "Diabetes",
 'pheno_ht_pre_cohort_start' : "Hypertension",
 'cardinal_symptoms': "Cardinal_symptomps",
 'flag_pre_cohort_start_exac_y1':  "Pre_baseline_exacerbation",
 'flag_pre_cohort_start_meds_ocs_y1': "Pre_baseline_OCS",
 'desc_smoking_at_baseline_Current': "Smoking_current",
 'desc_smoking_at_baseline_Previous': "Smoking_previous"
}


In [ ]:
df = df.rename(columns=rename_dict)

In [ ]:
cov_list = rename_dict.values()

In [ ]:
col_o

# Unadjusted Odds ratio

In [ ]:
# Prefered method (check table2x2, if any frequencies<5, repeat and use fisher as test type)
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= rename_dict.get("bmi_30_imputed"), col_o=col_o, test_type='chi-square')

In [ ]:
table2x2

In [ ]:
contingency_table = make_contingency_table(df, col_e=rename_dict.get("bmi_30_imputed"), col_o=col_o)
contingency_table

In [ ]:
def check_ci_overlap(ci1_lower, ci1_upper, ci2_lower, ci2_upper):
    """
    Checks the overlap of two 95% confidence intervals
    """
    if ci1_upper < ci2_lower or ci2_upper < ci1_lower:
        return "No overlap, significant difference"
    else:
        return "Overlap, no significant difference"

In [ ]:
np_random_seed

In [ ]:
dp_random_seed

# DP. Unadjusted OR

In [ ]:
# apply dp
epsilon = 0.05
total_count = round(dp.tools.count_nonzero(np.ones_like(df[col_o]), epsilon=epsilon, random_state=np_random_seed))
total_count

In [ ]:
contingency_table_dp = dp_contingency_table(df, col_o=col_o, 
                                            col_e=rename_dict.get("bmi_30_imputed"), 
                                            epsilon=epsilon, 
                                            random_state=dp_random_seed)
contingency_table_dp

In [ ]:
contingency_table_dp.sum().sum()

In [ ]:
dp_or, dp_ci_lower, dp_ci_upper, dp_p_value, table2x2= unadjusted_odds_ratio_calculator(df=None,
                                                                   col_e=rename_dict.get("bmi_30_imputed"), 
                                                                   col_o=col_o, 
                                                                   contingency_table=contingency_table_dp, 
                                                                   test_type='chi-square' )

In [ ]:
ratio_diff_pvalue(np_or, np_ci_lower, np_ci_upper, dp_or, dp_ci_lower, dp_ci_upper)

In [ ]:
overlap_trnval_np_dp = check_ci_overlap(np_ci_lower, np_ci_upper, dp_ci_lower, dp_ci_upper)
print(f"Model A: {overlap_trnval_np_dp}")

# BMI

In [ ]:
epsilons = [0.01, 0.025,0.05, 0.1, 0.25, 0.693, 1, 1.098, 2,5, 10]

In [ ]:
col_e = rename_dict.get("bmi_30_imputed")
col_e

In [ ]:
results_df_no_private= odds_ratio_across_epsilons_without_non_private_experimental(df,
                                                        col_e=rename_dict.get("bmi_30_imputed"), 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=2,
                                                        verbose=False, 
                                                         test_type='chi-square')
results_df_no_private

In [ ]:
epsilon_p_min= 0
for idx, row in results_df_no_private.iterrows():
    epsilon = row['epsilon']
    p_value = row['diff_p_value']
    if p_value >=0.05:
        epsilon_p_min= epsilon
        break

epsilon_p_min

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=rename_dict.get("bmi_30_imputed"), 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, 
                                                        test_type='chi-square')

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
             
                               title = "",
                               covariate_name_xlabel= "BMI≥30",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted", 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
results_df.loc[0]

In [ ]:
cols_list_all= ["Covariate", "or", 	"or_ci_lower", 	"or_ci_upper", "or_p_value"]
data_all = []

In [ ]:
col_e

In [ ]:
data_all.append({"Covariate": "BMI≥30", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Anxiety

In [ ]:
col_e = rename_dict.get("pheno_anxiety_pre_cohort_start")


In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, 
                                                        test_type='chi-square')

results_df

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df_no_private= odds_ratio_across_epsilons_without_non_private_experimental(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")

results_df_no_private

In [ ]:
epsilon_p_min= 0
for idx, row in results_df_no_private.iterrows():
    epsilon = row['epsilon']
    p_value = row['diff_p_value']
    if p_value >=0.05:
        epsilon_p_min= epsilon
        break
epsilon_p_min

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                       
                               title = "",
                               covariate_name_xlabel= "Anxiety",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False,
                               
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
data_all.append({"Covariate": "Anxiety", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# CKD

In [ ]:
col_e = rename_dict.get("pheno_ckd_pre_cohort_start")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df_no_private= odds_ratio_across_epsilons_without_non_private_experimental(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")
results_df_no_private

In [ ]:
epsilon_p_min= 0
for idx, row in results_df_no_private.iterrows():
    epsilon = row['epsilon']
    p_value = row['diff_p_value']
    if p_value >=0.05:
        epsilon_p_min= epsilon
        break

epsilon_p_min

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
        
                               title = "",
                               covariate_name_xlabel= "CKD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 15,
                               implausibly_high_text = "∞ (>15)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False
                               , 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
data_all.append({"Covariate": "CKD", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# COPD

In [ ]:
col_e = rename_dict.get("pheno_copd_pre_cohort_start")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                          
                               title = "",
                               covariate_name_xlabel= "COPD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=2, 
                               xlim_max=5, 
                               text_x_position=5.1
                          )

In [ ]:

data_all.append({"Covariate": "COPD", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# CVD

In [ ]:
col_e = rename_dict.get("pheno_cvd_pre_cohort_start")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                      
                               title = "",
                               covariate_name_xlabel= "CVD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 15,
                               implausibly_high_text = "∞ (>15)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:

data_all.append({"Covariate": "CVD", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Depression

In [ ]:
col_e = rename_dict.get("pheno_depression_pre_cohort_start")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                          
                               title = "",
                               covariate_name_xlabel= "Depression",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
data_all.append({"Covariate": "Depression", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Diabetes

In [ ]:
col_e = rename_dict.get("pheno_diabetes_pre_cohort_start")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                          
                               title = "",
                               covariate_name_xlabel= "Diabetes",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 15,
                               implausibly_high_text = "∞ (>15)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:

data_all.append({"Covariate": "Diabetes", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# HT

In [ ]:
col_e = rename_dict.get("pheno_ht_pre_cohort_start")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type='chi-square')

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
               
                               title = "",
                               covariate_name_xlabel= "Hypertension",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 15,
                               implausibly_high_text = "∞ (>15)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:

data_all.append({"Covariate": "Hypertension", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Cardinal symptoms

In [ ]:
col_e = rename_dict.get("cardinal_symptoms")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type="chi-square")

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                         
                               title = "",
                               covariate_name_xlabel= "Cardinal symptoms",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR\n",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=1.3, 
                               xlim_max=4.3, 
                               text_x_position=4.4
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                         
                               title = "",
                               covariate_name_xlabel= "Cardinal symptoms",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR\n",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=0, 
                               xlim_max=4, 
                               text_x_position=4.1, save_as_tiff="dp_unadjusted_cardinal"
                          )

In [ ]:
data_all.append({"Covariate": "Cardinal_symptomps", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Sex female

In [ ]:
col_e = rename_dict.get("sex_female")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type='chi-square')

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                              
                               title = "",
                               covariate_name_xlabel= "Female sex",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
data_all.append({"Covariate": "Sex_female", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Eth non white

In [ ]:
col_e = rename_dict.get("eth_non_white")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type='chi-square')

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                
                               title = "",
                               covariate_name_xlabel= "Non-white ethnicity",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR\n",
                               x_size=9, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.2, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
data_all.append({"Covariate": "Non_white", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Age 60+

In [ ]:
col_e = rename_dict.get("age_60+")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type='chi-square')

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                            
                               title = "",
                               covariate_name_xlabel= "Age≥60",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
data_all.append({"Covariate": "Age≥60", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Late onset

In [ ]:
col_e = rename_dict.get("late_onset_asthma_40")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type='chi-square')

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                        
                               title = "",
                               covariate_name_xlabel= "Late asthma onset",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=9, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
data_all.append({"Covariate": "Late_onset", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Current smoker

In [ ]:
col_e = rename_dict.get("desc_smoking_at_baseline_Current")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type='chi-square')

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                         
                               title = "",
                               covariate_name_xlabel= "Smoking-current",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=10, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
data_all.append({"Covariate": "Smoking_current", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Previous smoker

In [ ]:
col_e = rename_dict.get("desc_smoking_at_baseline_Previous")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type='chi-square')

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                       
                               title = "",
                               covariate_name_xlabel= "Smoking-previous",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=9, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
data_all.append({"Covariate": "Smoking_previous", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Pre Exac

In [ ]:
col_e = "flag_pre_cohort_start_exac_y1"
col_e = rename_dict.get("flag_pre_cohort_start_exac_y1")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type='chi-square')

results_df

In [ ]:

plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                       
                               title = "",
                               covariate_name_xlabel= "1-year exacerbation\nclinical",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=9, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=4.5, 
                               xlim_max=7.5, 
                               text_x_position=7.6
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                       
                               title = "",
                               covariate_name_xlabel= "1-year exacerbation\nclinical",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR\n",
                               x_size=9, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=1.2, 
                               xlim_max=7.8, 
                               text_x_position=7.9                          )

In [ ]:
data_all.append({"Covariate": "Pre_baseline_exacerbation", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# Pre OCS

In [ ]:
col_e = "flag_pre_cohort_start_meds_ocs_y1"
col_e = rename_dict.get("flag_pre_cohort_start_meds_ocs_y1")
col_e

In [ ]:
np_or, np_ci_lower, np_ci_upper, np_p_value, table2x2 = unadjusted_odds_ratio_calculator(df, col_e= col_e, col_o=col_o, test_type="chi-square")

In [ ]:
table2x2

In [ ]:
results_df = odds_ratio_across_epsilons_with_non_private(df,
                                                        col_e=col_e, 
                                                        col_o=col_o,
                                                        np_or=np_or,
                                                        np_ci_lower=np_ci_lower,
                                                        np_ci_upper=np_ci_upper,
                                                        np_p_value=np_p_value,
                                                        epsilons_list=epsilons,
                                                        random_state=dp_random_seed,
                                                        verbose=False, test_type='chi-square')

results_df

In [ ]:
plot_or_rr_vertical_version_44(results_df,
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                  
                               title = "",
                               covariate_name_xlabel= "1-year OCS\nprescription",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               custom_x_label="Unadjusted OR",
                               x_size=9, 
                               y_size=10,
                               adjustment="unadjusted",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=0.4, 
                               xlim_max=3.4, 
                               text_x_position=3.5
                          )

In [ ]:
data_all.append({"Covariate": "Pre_baseline_OCS", "or": results_df.loc[0][1], "or_ci_lower": results_df.loc[0][2], "or_ci_upper": results_df.loc[0][3], 
                "or_p_value": results_df.loc[0][4]})

# all non-private ORs

In [ ]:
def plot_forest_log_scale_method_222(df, type="RR", figsize=(10, 6), log_scale=True, 
                                     xtick_fontsize=10, xlim_max=None):
    """Forest plot of a single model in log scale with ORs/RRs, CIs, and p-values listed on the right."""
    sns.set(style="white")
    if type == "OR":
        ratio_text = 'or'
        ci_lower_text = 'or_ci_lower'
        ci_upper_text = 'or_ci_upper'
        p_value_text = 'or_p_value'
        title_text = "Unadjusted Odds"
    else:
        ratio_text = 'rr'
        ci_lower_text = 'rr_ci_lower'
        ci_upper_text = 'rr_ci_upper'
        p_value_text = 'rr_p_value'
        title_text = "Unadjusted Risk"
    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(df[ratio_text], df['Covariate'], color='none')
    ax.scatter(df[ratio_text], df['Covariate'], color='none')
    ax.invert_yaxis()
    for i, row in df.iterrows():
        fmt = 's'
        mfc_marker = 'white' if row[p_value_text] >= 0.05 else 'maroon'
        ax.errorbar(row[ratio_text], row['Covariate'],
                    xerr=[[row[ratio_text] - row[ci_lower_text]], [row[ci_upper_text] - row[ratio_text]]],
                    fmt=fmt, mfc=mfc_marker, color='maroon' if row[p_value_text] < 0.05 else 'maroon', 
                    ecolor='firebrick', capsize=0, markersize=4)
    ax.axvline(x=1, color='gray', linestyle=':', linewidth=1)
    if log_scale:
        ax.set_xscale('log')
    ax.set_xlabel(f'{title_text} Ratio {"(log scale)" if log_scale else ""}')
    ax.set_title(f'Forest Plot of {title_text} Ratios with 95% Confidence Intervals')
    ax.grid(True, linestyle='--', alpha=0.2)
    ax.tick_params(axis='y', which='major', labelsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(True)  
    ax.yaxis.set_ticks_position('none') 
    xlim = ax.get_xlim()
    x_text_position = xlim[1] * 1.0  
    for i, row in df.iterrows():
        ratio_value = f"{row[ratio_text]:.2f}"
        ci_text = f"({row[ci_lower_text]:.2f}, {row[ci_upper_text]:.2f})"
        if row[p_value_text] < 0.001:
            p_value_text_display = "p < 0.001"
        elif row[p_value_text] < 0.01:
            p_value_text_display = "p < 0.01"
        elif row[p_value_text] < 0.05:
            p_value_text_display = "p < 0.05"
        else:
            p_value_text_display = f"p= {row[p_value_text]:.2g}"
        y_value = row['Covariate']
        ax.text(x_text_position, y_value, 
                f"{ratio_value} {ci_text}", 
                ha='left', va='center', fontsize=9, color='black')
    if xlim_max is None:
        ax.set_xlim(left=xlim[0], right=xlim[1] * 1.2)
    else:
        ax.set_xlim(left=xlim[0], right=xlim_max)
    ax.xaxis.set_tick_params(labelsize=xtick_fontsize)
    ax.margins(y=0.1) 
    plt.tight_layout()
    plt.show()


In [ ]:
df_all2 = pd.DataFrame(data_all)

In [ ]:
df_all2

In [ ]:
col_order = ['Age≥60', 'Sex_female', 'Non_white', 'Late_onset', 'Anxiety', 'BMI≥30', 'CKD', 
             'COPD', 'CVD', 'Depression', 'Diabetes', 'Hypertension', 'Cardinal_symptomps', 'Pre_baseline_exacerbation', 
             'Pre_baseline_OCS', 'Pre_baseline_meds', 'Smoking_current', 'Smoking_previous', 'Near_major_road']
df_all2['Covariate'] = pd.Categorical(df_all2['Covariate'], categories=col_order, ordered=True)
df_all2 = df_all2.sort_values('Covariate')
df_all2 = df_all2.reset_index()

In [ ]:
df_all2 = df_all2.drop(columns=["index"])

In [ ]:
df_all2

In [ ]:
cov_selected = ['Age≥60', 
                'Sex_female', 
                'Non_white',  
                'Anxiety', 
                'BMI≥30', 
                'CKD', 
             'COPD', 
                'CVD', 
                'Diabetes', 
                'Hypertension',
                'Cardinal_symptomps', 
                'Pre_baseline_exacerbation',  
             'Pre_baseline_OCS']
df_all_3 = df_all2[df_all2["Covariate"].isin(cov_selected)]

In [ ]:
df_all_3['Covariate'] = df_all_3['Covariate'].replace('Sex_female', 'Female sex')
df_all_3['Covariate'] = df_all_3['Covariate'].replace('Non_white', 'Non white')
df_all_3['Covariate'] = df_all_3['Covariate'].replace('Pre_baseline_exacerbation', 'Pre exacerbation')
df_all_3['Covariate'] = df_all_3['Covariate'].replace('Pre_baseline_OCS', 'Pre OCS')                   

In [ ]:
# Full
plot_forest_log_scale_method_222(df_all_3, type="OR", figsize=(5, 5), log_scale=False, xlim_max=7.3)